# 01. Carga, Union y Limpieza

Este notebook muestra como se unieron los archivos de 2022, 2023 y 2024 usando `ocid` como llave principal.

## Librerias

In [ ]:
# Librerias principales para lectura, union y limpieza de datos.
from pathlib import Path
import pandas as pd
import numpy as np


## Rutas base

In [ ]:
# Definimos la carpeta donde estan los datos originales y donde guardaremos la base final.
BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
OUTPUT_FILE = DATA_DIR / 'contrataciones_peru_2022_2024_maestro.csv'
ANIOS = [2022, 2023, 2024]
print(DATA_DIR)


## Funciones de carga

In [ ]:
# Esta funcion lee el mismo archivo para cada anio y luego los apila en una sola tabla.
def apilar_archivo(nombre_archivo, columnas=None):
    partes = []
    for anio in ANIOS:
        ruta = DATA_DIR / str(anio) / nombre_archivo
        temp = pd.read_csv(ruta, usecols=columnas, low_memory=False)
        temp['anio'] = anio
        partes.append(temp)
    return pd.concat(partes, ignore_index=True)


## Carga de tablas necesarias

In [ ]:
# Cargamos solo las tablas que realmente usaremos en el proyecto.
main = apilar_archivo('main.csv', columnas=[
    'ocid', 'date', 'publishedDate', 'buyer_name', 'tender_title',
    'tender_mainProcurementCategory', 'tender_procurementMethod',
    'tender_numberOfTenderers', 'tender_value_amount_PEN', 'tender_datePublished'
])

parties = apilar_archivo('parties.csv', columnas=[
    'main_ocid', 'name', 'roles', 'address_department', 'address_region', 'address_locality'
])

tenderers = apilar_archivo('tender_tenderers.csv', columnas=['main_ocid', 'id'])
awards = apilar_archivo('awards.csv', columnas=['main_ocid', 'value_amount', 'value_currency'])
contracts_items = apilar_archivo('contracts_items.csv', columnas=['main_ocid', 'id', 'classification_description'])
awards_suppliers = apilar_archivo('awards_suppliers.csv', columnas=['main_ocid', 'name'])

print('main:', main.shape)
print('parties:', parties.shape)
print('tenderers:', tenderers.shape)
print('awards:', awards.shape)
print('contracts_items:', contracts_items.shape)
print('awards_suppliers:', awards_suppliers.shape)


## Resumen de tablas auxiliares

In [ ]:
# Filtramos solo las entidades compradoras para obtener departamento y nombre de la entidad.
buyers = parties[parties['roles'].fillna('').str.contains('buyer|procuringEntity', case=False, regex=True)].copy()
buyers = buyers.groupby('main_ocid', as_index=False).agg(
    entidad_compradora=('name', 'first'),
    departamento=('address_department', 'first'),
    region=('address_region', 'first'),
    localidad=('address_locality', 'first')
).rename(columns={'main_ocid': 'ocid'})

# Contamos postores unicos por proceso.
resumen_postores = tenderers.groupby('main_ocid', as_index=False).agg(
    n_postores_real=('id', 'nunique')
).rename(columns={'main_ocid': 'ocid'})

# Sumamos el monto adjudicado por proceso.
resumen_awards = awards.groupby('main_ocid', as_index=False).agg(
    monto_adjudicado=('value_amount', 'sum')
).rename(columns={'main_ocid': 'ocid'})

# Contamos items y guardamos una clasificacion referencial.
resumen_items = contracts_items.groupby('main_ocid', as_index=False).agg(
    n_items=('id', 'nunique'),
    clasificacion=('classification_description', 'first')
).rename(columns={'main_ocid': 'ocid'})

# Guardamos un proveedor ganador referencial por proceso.
resumen_proveedor = awards_suppliers.groupby('main_ocid', as_index=False).agg(
    proveedor_ganador=('name', 'first')
).rename(columns={'main_ocid': 'ocid'})

buyers.head()


## Union del DataFrame maestro

In [ ]:
# Partimos desde main y hacemos merges por ocid.
df = main.copy()
df = df.merge(buyers, on='ocid', how='left')
df = df.merge(resumen_postores, on='ocid', how='left')
df = df.merge(resumen_awards, on='ocid', how='left')
df = df.merge(resumen_items, on='ocid', how='left')
df = df.merge(resumen_proveedor, on='ocid', how='left')

print(df.shape)
df.head()


## Limpieza y variables derivadas

In [ ]:
# Convertimos fechas a formato fecha para poder agrupar por mes y anio.
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['publishedDate'] = pd.to_datetime(df['publishedDate'], errors='coerce')
df['tender_datePublished'] = pd.to_datetime(df['tender_datePublished'], errors='coerce')
df['fecha_proceso'] = df['tender_datePublished'].fillna(df['date']).fillna(df['publishedDate'])
df['fecha_proceso'] = df['fecha_proceso'].dt.tz_localize(None)
df['mes'] = df['fecha_proceso'].dt.to_period('M').astype(str)
df['semana'] = df['fecha_proceso'].dt.isocalendar().week.astype('Int64')

# Estandarizamos variables de categoria y metodo.
mapa_categoria = {'goods': 'Bienes', 'services': 'Servicios', 'works': 'Obras'}
df['categoria'] = df['tender_mainProcurementCategory'].astype(str).str.lower().map(mapa_categoria).fillna('Sin dato')

mapa_metodo = {'open': 'Competitivo', 'selective': 'Selectivo', 'limited': 'Directa', 'direct': 'Directa'}
df['metodo_simple'] = df['tender_procurementMethod'].astype(str).str.lower().map(mapa_metodo).fillna('Sin dato')

# Definimos el numero final de postores y el indicador de un solo postor.
df['tender_numberOfTenderers'] = pd.to_numeric(df['tender_numberOfTenderers'], errors='coerce')
df['n_postores'] = df['n_postores_real'].fillna(df['tender_numberOfTenderers'])
df['un_solo_postor'] = df['n_postores'] == 1

# El monto principal del tablero sera el monto adjudicado.
df['monto_adjudicado'] = pd.to_numeric(df['monto_adjudicado'], errors='coerce').fillna(0)
df['monto_MM'] = df['monto_adjudicado'] / 1_000_000

# Rellenamos nulos en textos para que los graficos no fallen.
for col in ['entidad_compradora', 'departamento', 'region', 'localidad', 'clasificacion', 'proveedor_ganador']:
    df[col] = df[col].fillna('Sin dato')

df[['ocid', 'anio', 'categoria', 'metodo_simple', 'n_postores', 'un_solo_postor', 'monto_adjudicado']].head()


## Guardar la base final

In [ ]:
# Exportamos solo a CSV para mantener el proyecto mas simple y facil de explicar.
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print('Archivo guardado en:')
print(OUTPUT_FILE)
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
